# validate_kev
Bronze KEV는 이제 별도 Function이 GitHub 변경 감지 / KEV 파일 Fetch / Hash 비교까지 끝내고, `event_type`(`kev_added`/`kev_updated`/`kev_removed_confirmed`) 단위로 정규화된 이벤트 파일을 `Files/bronze/kev`에 쌓아준다. 파일명 자체가 `{run_at}_{commit_sha[:7]}.json`로 이미 결정적(deterministic)이다.

**Fabric은 GitHub API를 다시 호출하지 않는다** — Function과 역할이 겹치므로, `Files/bronze/kev` 폴더를 읽기만 한다.

- **Fatal**: `cve_id` 없음, 또는 `event_type`이 `kev_added`/`kev_updated`/`kev_removed_confirmed` 중 하나가 아님 → Reject.
- **Watermark 기반 증분 읽기**: KEV Bronze는 날짜 폴더 없이 계속 누적되는 Event Log이므로 매번 폴더 전체를 다시 읽지 않는다. `run_mode == "BACKFILL"`이면 `target_cve_ids`로 스코프를 좁혀 전체 이력을 읽고(워터마크 무시), 그 외에는 `meta_watermark`의 마지막 처리 지점 이후 파일만 읽는다.
- **`_meta.json` 없음**: 파일명이 이미 `run_at`+`commit_sha`를 담고 있어 별도 메타 파일이 필요 없다. 이 값을 그대로 **결정적 `batch_id`**로 쓴다 — 같은 이벤트를 재처리(Replay)해도 항상 같은 `batch_id`가 나온다.

In [ ]:
%run 00_silver_common

In [ ]:
raw_df = spark.read.option("multiLine", "true").json(f"{BRONZE_KEV_PATH}/*")
raw_df = (
    raw_df
    .withColumn("_file", F.input_file_name())
    .withColumn("_file_stem", F.regexp_extract(F.col("_file"), r"([^/\\]+)\.json$", 1))
    .withColumn("_run_at", F.regexp_extract(F.col("_file_stem"), r"^([^_]+)_", 1))
)
raw_source_cols = [c for c in raw_df.columns if not c.startswith("_")]


In [ ]:
if run_mode == "BACKFILL":
    if not target_cve_ids:
        raise ValueError("BACKFILL 모드는 target_cve_ids가 필요합니다.")
    scoped_df = raw_df.filter(F.upper(F.trim(existing_col(raw_df, "cve_id"))).isin(target_cve_ids))
    watermark_source = None  # BACKFILL은 CVE 스코프라 워터마크를 안 건드린다
else:
    wm = get_watermark("cisa_kev")
    last_processed_run_at = wm["last_processed_timestamp"] if wm else None
    scoped_df = raw_df.filter(F.col("_run_at") > last_processed_run_at) if last_processed_run_at else raw_df
    watermark_source = "cisa_kev"

scoped_count = scoped_df.count()


In [ ]:
VALID_EVENT_TYPES = ["kev_added", "kev_updated", "kev_removed_confirmed"]

cve_id_candidate = F.upper(F.trim(existing_col(scoped_df, "cve_id")))
event_type_candidate = existing_col(scoped_df, "event_type")

scoped_df = (
    scoped_df
    .withColumn("_cve_id_candidate", cve_id_candidate)
    .withColumn("_event_type_candidate", event_type_candidate)
)
fatal_cond = (
    F.col("_cve_id_candidate").isNull()
    | F.col("_event_type_candidate").isNull()
    | (~F.col("_event_type_candidate").isin(VALID_EVENT_TYPES))
)


In [ ]:
reject_df = (
    scoped_df.filter(fatal_cond)
    .withColumn(
        "reject_reason",
        F.when(F.col("_cve_id_candidate").isNull(), F.lit("MISSING_CVE_ID"))
         .otherwise(F.lit("INVALID_EVENT_TYPE")),
    )
    .withColumn("raw_payload", F.to_json(F.struct(*raw_source_cols)))
    .withColumn("source", F.lit("cisa_kev"))
    .withColumn("batch_id", F.col("_file_stem"))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
    .select("raw_payload", "reject_reason", "source", "batch_id", "ingested_at")
)

valid_df = (
    scoped_df.filter(~fatal_cond)
    .withColumnRenamed("_cve_id_candidate", "cve_id_raw")
    .withColumnRenamed("_event_type_candidate", "event_type")
    .withColumn("batch_id", F.col("_file_stem"))  # 결정적 batch_id = 파일명(run_at_commitsha)
    .withColumn("ingested_at", F.lit(INGESTED_AT))
    .drop("_file", "_run_at")
)

reject_count = reject_df.count()
valid_count = valid_df.count()

if reject_count > 0:
    write_append(reject_df, TBL["reject_kev"])
write_overwrite(valid_df, TBL["stg_kev_valid"])


In [ ]:
# 이번에 실제로 읽은 파일들 중 가장 최신 것으로 워터마크를 전진시킨다 (BACKFILL은 워터마크 미적용).
if watermark_source and scoped_count > 0:
    max_row = scoped_df.orderBy(F.col("_run_at").desc()).select("_run_at", "_file_stem").first()
    if max_row is not None and max_row["_run_at"]:
        update_watermark(watermark_source, max_row["_file_stem"], max_row["_run_at"])

log_batch_run("validate_kev", "SUCCESS", record_count=valid_count)
log_dq_metric("validate_kev", "reject_rate", reject_count / max(valid_count + reject_count, 1), passed=True,
              detail=f"valid={valid_count}, reject={reject_count}, run_mode={run_mode}, files_read={scoped_count}")
print(f"[validate_kev] valid={valid_count} reject={reject_count} files_read={scoped_count} "
      f"run_mode={run_mode} batch_id(run)={batch_id}")
